# Experiment Notebook — SimpleCNN Multiclass Hyperparameter Search
**D7047E Advanced Deep Learning | Group 14**

Based on Karpathy's recipe:
1. Overfit a single batch → verify model/loss/optimizer are correct
2. Train on small subset (~1000 samples) → tune LR, batch size, regularization
3. Move findings to `01_simplecnn_multiclass.ipynb` for full run

**Do not use checkpoints from this notebook for final evaluation.**

## 1. Configuration — edit these to experiment

In [ ]:
import sys
sys.path.insert(0, '..')
DATA_DIR       = '../dataset/iam_crossouts'
CHECKPOINT_DIR = '../checkpoints'
ZIP_PATH       = '../dataset/adl_dataset.zip'
FILE_ID        = '1dgIfz8aFwCuphLN9-L7gcU4QN0UQEKP3'
IMG_SIZE       = 224
MODEL          = 'SimpleCNN-Experiment'

# --- Hyperparameters to tune ---
LR         = 1e-4    # try: 1e-3, 1e-4, 5e-4
BATCH_SIZE = 64      # try: 32, 64, 128
EPOCHS     = 50
MIN_EPOCHS = 10
PATIENCE   = 10
NUM_WORKERS = 16

# --- Subset sizes (Karpathy recipe) ---
# Stage 1: overfit one batch → set TRAIN_SIZE = BATCH_SIZE, VAL_SIZE = BATCH_SIZE
# Stage 2: small subset → set TRAIN_SIZE = 1000, VAL_SIZE = 200
# Stage 3: full data → set both to None
TRAIN_SIZE = BATCH_SIZE   # Stage 1 default
VAL_SIZE   = BATCH_SIZE

print(f'LR={LR}  BATCH={BATCH_SIZE}  TRAIN={TRAIN_SIZE}  VAL={VAL_SIZE}')

## 2. Setup

In [ ]:
!pip install -q gdown torch torchvision pillow matplotlib scikit-learn wandb python-dotenv

In [ ]:
import os, zipfile, gc
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score
import wandb
from dotenv import load_dotenv

from common import (get_transforms, WANDB_PROJECT, WANDB_GROUP_MULTICLASS,
                    CrossOutDataset, train_model, CATEGORIES, SimpleCNN)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
load_dotenv()
wandb.login(key=os.environ.get('WANDB_API_KEY'))
torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 3. Dataset Download

In [ ]:
try:
    import gdown
except ImportError:
    import subprocess; subprocess.run(['pip','install','gdown','-q'],check=True); import gdown
os.makedirs(os.path.dirname(ZIP_PATH), exist_ok=True)
if not os.path.exists(ZIP_PATH):
    gdown.download(f'https://drive.google.com/uc?id={FILE_ID}', ZIP_PATH, quiet=False)
if not os.path.exists(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH,'r') as zf: zf.extractall(DATA_DIR)
print('Dataset ready.')

## 4. Data Loaders (subset)

In [ ]:
train_t, val_t = get_transforms(IMG_SIZE)

mc_train_full = CrossOutDataset(os.path.join(DATA_DIR,'train','images'), CATEGORIES, train_t)
mc_val_full   = CrossOutDataset(os.path.join(DATA_DIR,'val',  'images'), CATEGORIES, val_t)

mc_train = Subset(mc_train_full, range(TRAIN_SIZE)) if TRAIN_SIZE else mc_train_full
mc_val   = Subset(mc_val_full,   range(VAL_SIZE))   if VAL_SIZE   else mc_val_full

ldr_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
              pin_memory=True, prefetch_factor=2 if NUM_WORKERS>0 else None)
train_loader = DataLoader(mc_train, shuffle=True,  **ldr_kw)
val_loader   = DataLoader(mc_val,   shuffle=False, **ldr_kw)
print(f'Train: {len(mc_train):,}  Val: {len(mc_val):,}')

## 5. Model

In [ ]:
model = SimpleCNN(num_classes=len(CATEGORIES))
tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'SimpleCNN  trainable params: {tr:,}')

## 6. Train

**Stage 1 — overfit one batch:** train_loss should reach ~0.0. If not, the model/loss is broken.

**Stage 2 — small subset:** tune LR and regularization. Look for smooth val_loss without wild swings.

**Stage 3 — full data:** copy best hyperparameters to `01_simplecnn_multiclass.ipynb`.

In [ ]:
save_path = os.path.join(CHECKPOINT_DIR, f'exp_{MODEL}.pth')

model, history = train_model(
    MODEL, model, train_loader, val_loader,
    nn.CrossEntropyLoss(),
    task='multiclass', save_path=save_path, device=device,
    lr=LR, epochs=EPOCHS, min_epochs=MIN_EPOCHS, patience=PATIENCE,
    wandb_project=WANDB_PROJECT, wandb_group='experiment', batch_size=BATCH_SIZE,
)

## 7. Quick Evaluation

In [ ]:
model.eval()
preds, labels = [], []
with torch.no_grad():
    for imgs, lbs in val_loader:
        preds.extend(model(imgs.to(device)).argmax(1).cpu().tolist())
        labels.extend(lbs.tolist())

print(f'Val Accuracy: {accuracy_score(labels, preds):.4f}')
print(f'Val Macro-F1: {f1_score(labels, preds, average="macro", zero_division=0):.4f}')

## 8. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train'); ax1.plot(history['val_loss'], label='Val')
ax1.set_title(f'Loss — LR={LR} BS={BATCH_SIZE} N={TRAIN_SIZE}'); ax1.legend()
ax2.plot(history['train_acc'], label='Train'); ax2.plot(history['val_acc'], label='Val')
ax2.set_title(f'Accuracy — LR={LR} BS={BATCH_SIZE} N={TRAIN_SIZE}'); ax2.legend()
plt.tight_layout()
plt.savefig(f'exp_lr{LR}_bs{BATCH_SIZE}_n{TRAIN_SIZE}.png', dpi=150)
plt.show()
del model; torch.cuda.empty_cache(); gc.collect()

---
## Findings Log
Record your observations here after each experiment:

| LR | Batch | Train N | Result | Notes |
|---|---|---|---|---|
| 1e-4 | 64 | 64 | ? | Stage 1 overfit test |


---
## Random Hyperparameter Search (Karpathy-style)

Search LR and weight decay in **log space** with random sampling.
Train each config for a few epochs on a small subset, then plot val_acc vs each hyperparameter.

> "Do not use grid search. Use random search." — Karpathy, CS231n

In [ ]:
import numpy as np

# Search space
N_TRIALS    = 20       # number of random configs to try
SEARCH_EPOCHS = 10     # epochs per trial (short — just enough to see the trend)
SEARCH_TRAIN  = 1000   # samples for search
SEARCH_VAL    = 200

# Log-uniform sampling (Karpathy: always search LR in log space)
np.random.seed(42)
search_configs = [
    {
        'lr':           10 ** np.random.uniform(-5, -2),
        'batch_size':   int(np.random.choice([32, 64, 128, 256])),
        'weight_decay': 10 ** np.random.uniform(-6, -2),
    }
    for _ in range(N_TRIALS)
]

print(f'Running {N_TRIALS} random trials, {SEARCH_EPOCHS} epochs each on {SEARCH_TRAIN} samples')
for i, cfg in enumerate(search_configs):
    print(f'  [{i+1:2d}] lr={cfg["lr"]:.2e}  bs={cfg["batch_size"]}  wd={cfg["weight_decay"]:.2e}')

In [ ]:
import torch.optim as optim
from sklearn.metrics import accuracy_score

results = []

train_t, val_t = get_transforms(IMG_SIZE)
mc_train_full = CrossOutDataset(os.path.join(DATA_DIR,'train','images'), CATEGORIES, train_t)
mc_val_full   = CrossOutDataset(os.path.join(DATA_DIR,'val',  'images'), CATEGORIES, val_t)
mc_train_s = Subset(mc_train_full, range(SEARCH_TRAIN))
mc_val_s   = Subset(mc_val_full,   range(SEARCH_VAL))

for i, cfg in enumerate(search_configs):
    print(f'\n[{i+1}/{N_TRIALS}] lr={cfg["lr"]:.2e}  bs={cfg["batch_size"]}  wd={cfg["weight_decay"]:.2e}')

    ldr_kw = dict(batch_size=cfg['batch_size'], num_workers=NUM_WORKERS,
                  pin_memory=True, prefetch_factor=2)
    tr_loader = DataLoader(mc_train_s, shuffle=True,  **ldr_kw)
    vl_loader = DataLoader(mc_val_s,   shuffle=False, **ldr_kw)

    m = SimpleCNN(num_classes=len(CATEGORIES)).to(device)
    opt = optim.Adam(m.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    criterion = nn.CrossEntropyLoss()
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    best_val_acc = 0.0
    for epoch in range(SEARCH_EPOCHS):
        m.train()
        for imgs, lbls in tr_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt.zero_grad()
            with torch.amp.autocast('cuda', enabled=use_amp):
                loss = criterion(m(imgs), lbls)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()

        m.eval()
        preds, labels = [], []
        with torch.no_grad():
            for imgs, lbls in vl_loader:
                preds.extend(m(imgs.to(device)).argmax(1).cpu().tolist())
                labels.extend(lbls.tolist())
        val_acc = accuracy_score(labels, preds)
        best_val_acc = max(best_val_acc, val_acc)

    results.append({**cfg, 'val_acc': best_val_acc})
    print(f'  best val_acc: {best_val_acc:.4f}')
    del m; torch.cuda.empty_cache(); gc.collect()

print('\nSearch complete.')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

lrs      = [r['lr']           for r in results]
wds      = [r['weight_decay'] for r in results]
bss      = [r['batch_size']   for r in results]
val_accs = [r['val_acc']      for r in results]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# LR vs val_acc (log scale — Karpathy: always plot LR on log axis)
axes[0].scatter(lrs, val_accs, c=val_accs, cmap='RdYlGn', s=80, edgecolors='k')
axes[0].set_xscale('log')
axes[0].set_xlabel('Learning Rate (log scale)')
axes[0].set_ylabel('Val Accuracy')
axes[0].set_title('LR vs Val Accuracy')

# Weight decay vs val_acc (log scale)
axes[1].scatter(wds, val_accs, c=val_accs, cmap='RdYlGn', s=80, edgecolors='k')
axes[1].set_xscale('log')
axes[1].set_xlabel('Weight Decay (log scale)')
axes[1].set_title('Weight Decay vs Val Accuracy')

# Batch size vs val_acc
axes[2].scatter(bss, val_accs, c=val_accs, cmap='RdYlGn', s=80, edgecolors='k')
axes[2].set_xlabel('Batch Size')
axes[2].set_title('Batch Size vs Val Accuracy')

plt.suptitle('Random Hyperparameter Search — SimpleCNN Multiclass', fontsize=13)
plt.tight_layout()
plt.savefig('random_search_results.png', dpi=150)
plt.show()

# Print top 5 configs
print('\nTop 5 configs:')
top = sorted(results, key=lambda x: x['val_acc'], reverse=True)[:5]
for r in top:
    print(f"  val_acc={r['val_acc']:.4f}  lr={r['lr']:.2e}  bs={r['batch_size']}  wd={r['weight_decay']:.2e}")

---
## Debug — Inspect Val Set at Epoch 9

To use this: add `DEBUG_EPOCHS = [9]` to the training loop in `common/__init__.py` temporarily,
or run inference manually by loading a checkpoint and evaluating the val set epoch-by-epoch.

In [ ]:
# Debug: train epoch-by-epoch and capture val predictions + confusion matrix at epoch 9
# Set DEBUG_EPOCH to whichever epoch you want to inspect

DEBUG_EPOCH = 9
DEBUG_TRAIN = 1000   # use small subset for speed
DEBUG_VAL   = 200
DEBUG_LR    = 1e-4
DEBUG_BS    = 64

train_t, val_t = get_transforms(IMG_SIZE)
mc_train_full = CrossOutDataset(os.path.join(DATA_DIR,'train','images'), CATEGORIES, train_t)
mc_val_full   = CrossOutDataset(os.path.join(DATA_DIR,'val',  'images'), CATEGORIES, val_t)

tr_loader = DataLoader(Subset(mc_train_full, range(DEBUG_TRAIN)), batch_size=DEBUG_BS,
                       shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, prefetch_factor=2)
vl_loader = DataLoader(Subset(mc_val_full,   range(DEBUG_VAL)),  batch_size=DEBUG_BS,
                       shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, prefetch_factor=2)

debug_model = SimpleCNN(num_classes=len(CATEGORIES)).to(device)
debug_opt   = torch.optim.Adam(debug_model.parameters(), lr=DEBUG_LR)
criterion   = nn.CrossEntropyLoss()
use_amp     = device.type == 'cuda'
scaler      = torch.amp.GradScaler('cuda', enabled=use_amp)

epoch_stats = []

for epoch in range(1, DEBUG_EPOCH + 2):  # run one epoch past the spike
    debug_model.train()
    for imgs, lbls in tr_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        debug_opt.zero_grad()
        with torch.amp.autocast('cuda', enabled=use_amp):
            loss = criterion(debug_model(imgs), lbls)
        scaler.scale(loss).backward()
        scaler.step(debug_opt); scaler.update()

    debug_model.eval()
    preds, labels, losses = [], [], []
    with torch.no_grad():
        for imgs, lbls in vl_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            with torch.amp.autocast('cuda', enabled=use_amp):
                out = debug_model(imgs)
                l   = criterion(out, lbls)
            preds.extend(out.argmax(1).cpu().tolist())
            labels.extend(lbls.cpu().tolist())
            losses.append(l.item())

    val_acc  = accuracy_score(labels, preds)
    val_loss = sum(losses) / len(losses)
    epoch_stats.append({'epoch': epoch, 'val_acc': val_acc, 'val_loss': val_loss,
                        'preds': preds[:], 'labels': labels[:]})
    print(f'Epoch {epoch:2d} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}')

del debug_model; torch.cuda.empty_cache(); gc.collect()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Plot confusion matrices for epoch before spike, at spike, and after
epochs_to_plot = [max(1, DEBUG_EPOCH - 1), DEBUG_EPOCH, DEBUG_EPOCH + 1]
fig, axes = plt.subplots(1, len(epochs_to_plot), figsize=(18, 5))

for ax, ep in zip(axes, epochs_to_plot):
    stat = next((s for s in epoch_stats if s['epoch'] == ep), None)
    if stat is None:
        continue
    cm = confusion_matrix(stat['labels'], stat['preds'])
    ConfusionMatrixDisplay(cm, display_labels=CATEGORIES).plot(
        ax=ax, xticks_rotation=45, colorbar=False, cmap='Blues')
    ax.set_title(f'Epoch {ep} | val_acc={stat["val_acc"]:.3f} | val_loss={stat["val_loss"]:.3f}')

plt.suptitle(f'Confusion Matrix around epoch {DEBUG_EPOCH} spike', fontsize=13)
plt.tight_layout()
plt.savefig(f'debug_epoch{DEBUG_EPOCH}_confusion.png', dpi=150)
plt.show()

# Show per-class accuracy at spike epoch
spike_stat = next(s for s in epoch_stats if s['epoch'] == DEBUG_EPOCH)
cm = confusion_matrix(spike_stat['labels'], spike_stat['preds'])
print(f'\nPer-class accuracy at epoch {DEBUG_EPOCH}:')
for i, cat in enumerate(CATEGORIES):
    total = cm[i].sum()
    correct = cm[i][i]
    print(f'  {cat:15s}: {correct}/{total} = {correct/total:.3f}' if total > 0 else f'  {cat}: no samples')